<a href="https://colab.research.google.com/github/sadumina/Deep-Learning-Assignment-Group-ID-5/blob/model%2FClassic-CNN/TrainNewCnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Connect Drive To Get the Dataset Access*

In [6]:
from google.colab import drive
drive.mount('/content/drive')

# find the exact zip path inside your Drive
!find /content/drive/MyDrive -iname "gaussian_filtered_images.zip" 2>/dev/null

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/data/gaussian_filtered_images.zip


*unzip the folder from Drive*

In [9]:
!unzip -q /content/gaussian_filtered_images.zip -d /content/dataset

Discover the Datasets Classes Using Python cmd

In [10]:
import os
for root, dirs, files in os.walk('/content/dataset'):
    print(root, '->', len(files), 'files,', dirs)
    if root.count('/') > 4:
        break

/content/dataset -> 0 files, ['gaussian_filtered_images']
/content/dataset/gaussian_filtered_images -> 1 files, ['Moderate', 'Severe', 'No_DR', 'Proliferate_DR', 'Mild']
/content/dataset/gaussian_filtered_images/Moderate -> 999 files, []
/content/dataset/gaussian_filtered_images/Severe -> 193 files, []
/content/dataset/gaussian_filtered_images/No_DR -> 1805 files, []
/content/dataset/gaussian_filtered_images/Proliferate_DR -> 295 files, []
/content/dataset/gaussian_filtered_images/Mild -> 370 files, []


*export pkl file to content*

In [11]:
import os
root_files = [f for f in os.listdir('/content/dataset/gaussian_filtered_images')
              if os.path.isfile(os.path.join('/content/dataset/gaussian_filtered_images', f))]
print(root_files)

['export.pkl']


*sorted Classes Names as include in Datasets*

In [12]:
DATA_DIR = '/content/dataset/gaussian_filtered_images'
class_names = sorted(['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'])
print(class_names)

['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']


*Splitting datasets into training and Validation and Testing*

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

filepaths, labels = [], []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_path):
        filepaths.append(os.path.join(cls_path, fname))
        labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
print(df['label'].value_counts())

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=42
)
print("\nTrain distribution:\n", train_df['label'].value_counts())
print("\nVal distribution:\n", val_df['label'].value_counts())

label
No_DR             1805
Moderate           999
Mild               370
Proliferate_DR     295
Severe             193
Name: count, dtype: int64

Train distribution:
 label
No_DR             1534
Moderate           849
Mild               314
Proliferate_DR     251
Severe             164
Name: count, dtype: int64

Val distribution:
 label
No_DR             271
Moderate          150
Mild               56
Proliferate_DR     44
Severe             29
Name: count, dtype: int64


*Add a Class weight for equally distributions*

In [43]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Extract labels from your training dataset
train_labels = np.concatenate([y for x, y in train_ds], axis=0)

# Calculate balancing weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print("Your Class Weights:", class_weight_dict)


Your Class Weights: {0: np.float64(1.9821656050955414), 1: np.float64(0.7330977620730271), 2: np.float64(0.4057366362451108), 3: np.float64(2.4796812749003982), 4: np.float64(3.795121951219512)}


*Data extract into Implementation*

In [15]:
import os, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

DATA_DIR = '/content/dataset/gaussian_filtered_images'
class_names = sorted(['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'])

filepaths, labels = [], []
for cls in class_names:
    cls_path = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_path):
        filepaths.append(os.path.join(cls_path, fname))
        labels.append(cls)

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

label_to_index = {name: i for i, name in enumerate(class_names)}
train_labels_idx = train_df['label'].map(label_to_index).values
class_weights = compute_class_weight('balanced', classes=np.unique(train_labels_idx), y=train_labels_idx)
class_weight_dict = dict(enumerate(class_weights))
print(class_weight_dict)

{0: np.float64(1.9821656050955414), 1: np.float64(0.7330977620730271), 2: np.float64(0.4057366362451108), 3: np.float64(2.4796812749003982), 4: np.float64(3.795121951219512)}


*Add Data augmentation layer for recaling Images or the Dataset*

In [28]:
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
num_classes = len(class_names)

def load_image(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0     # <-- must happen here, on the base image
    return img, label

def make_dataset(df, training):
    paths = df['filepath'].values
    labels = df['label'].map(label_to_index).values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
    layers.RandomContrast(0.1),
])

def augment_and_clip(image, label):
    image = data_augmentation(image, training=True)
    image = tf.clip_by_value(image, 0.0, 1.0)   # <-- force back into valid range
    return image, label

train_ds = train_ds.map(augment_and_clip, num_parallel_calls=tf.data.AUTOTUNE)

*testing Pixel Values ranges *

In [25]:
for imgs, labels in train_ds.take(1):
    print("min/max pixel values:", imgs.numpy().min(), imgs.numpy().max())

min/max pixel values: 0.0 1.0


***Build Convoulution Layer With 5 Layers***

In [42]:
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Fixed custom architecture with corrected spatial downsampling
model = Sequential([
    layers.Input(shape=(299, 299, 3)),

    layers.Conv2D(32, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(), layers.Dropout(0.2),

    layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(), layers.Dropout(0.25),

    layers.Conv2D(128, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(), layers.Dropout(0.25),

    layers.Conv2D(256, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(), layers.Dropout(0.3),

    layers.Conv2D(512, 3, padding='same', kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.MaxPooling2D(), # <-- CRITICAL FIX: Smoothly downsamples 18x18 to 9x9 before GAP
    layers.Dropout(0.4),

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4)),
    layers.BatchNormalization(), layers.Activation('relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

# Compiled with a faster initial learning rate (1e-3) to boost the 41% accuracy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Your excellent callback strategy (adjusted patience slightly to match 1e-3 training pace)
callbacks_phase1 = [
    ModelCheckpoint('/content/drive/MyDrive/dr_project/checkpoints/phase1_best.keras',
                     monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_phase1
)


Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step - accuracy: 0.3503 - loss: 1.9239
Epoch 1: val_loss improved from None to 1.41698, saving model to /content/drive/MyDrive/dr_project/checkpoints/phase1_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/dr_project/checkpoints/phase1_best.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 65s 486ms/step - accuracy: 0.4007 - loss: 1.7357 - val_accuracy: 0.4927 - val_loss: 1.4170 - learning_rate: 0.0010
Epoch 2/30
97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step - accuracy: 0.4171 - loss: 1.6294
Epoch 2: val_loss improved from 1.41698 to 1.40186, saving model to /content/drive/MyDrive/dr_project/checkpoints/phase1_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/dr_project/checkpoints/phase1_best.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 38s 367ms/step - accuracy: 0.4174 - loss: 1.6214 - val_accuracy: 0.4927 - val_loss: 1.4019 - learning_rate: 0.0010
Epoch 3/30
97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step - accuracy: 0.4226 - loss: 1.5780

In [35]:
print(class_weight_dict)

{0: np.float64(1.9821656050955414), 1: np.float64(0.7330977620730271), 2: np.float64(0.4057366362451108), 3: np.float64(2.4796812749003982), 4: np.float64(3.795121951219512)}


In [18]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

drive.mount('/content/drive')  # if not already mounted
os.makedirs('/content/drive/MyDrive/dr_project/checkpoints', exist_ok=True)

callbacks = [
    ModelCheckpoint('/content/drive/MyDrive/dr_project/checkpoints/best_model.keras',
                     monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
model.optimizer.learning_rate.assign(2e-5)  # reset LR since it decayed down to 5e-6 last time

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=callbacks_phase2
)

Epoch 1/15
97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 363ms/step - accuracy: 0.3859 - loss: 2.3131
Epoch 1: val_loss improved from 1.31151 to 1.29001, saving model to /content/drive/MyDrive/dr_project/checkpoints/phase2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/dr_project/checkpoints/phase2_best.keras
98/98 ━━━━━━━━━━━━━━━━━━━━ 39s 375ms/step - accuracy: 0.3888 - loss: 2.2089 - val_accuracy: 0.5364 - val_loss: 1.2900 - learning_rate: 2.0000e-05
Epoch 2/15
97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 346ms/step - accuracy: 0.3757 - loss: 2.1327
Epoch 2: val_loss did not improve from 1.29001
98/98 ━━━━━━━━━━━━━━━━━━━━ 38s 357ms/step - accuracy: 0.3699 - loss: 2.1528 - val_accuracy: 0.5418 - val_loss: 1.2921 - learning_rate: 2.0000e-05
Epoch 3/15
97/98 ━━━━━━━━━━━━━━━━━━━━ 0s 359ms/step - accuracy: 0.3556 - loss: 2.1023
Epoch 3: val_loss did not improve from 1.29001
98/98 ━━━━━━━━━━━━━━━━━━━━ 39s 369ms/step - accuracy: 0.3483 - loss: 2.1099 - val_accuracy: 0.4927 - val_loss: 1.3095 - learn